# RMD (PSU Research Metadata Database)

Walks every RMD endpoint the pipeline pulls from, with a small sample for each.  RMD is PSU's internal research catalog (HHD orgs only at this point — full-university access is pending).

**Base URL:** `https://metadata.libraries.psu.edu/v1`  
**Auth:** `X-API-Key` header for most endpoints; user `/profile` is public.

What it gives the pipeline: the **researcher cohort** (who's at PSU + their PSU IDs), grants, presentations, ETDs, org memberships, and the publication-contributor scan we use to build the ORCID→WebAccess map.

## Setup
API key from AWS Secrets Manager via the instance role; falls back to `PSU_RESEARCH_API_KEY` env var.

In [ ]:
import json, os, time
import requests

try:
    import boto3
    sm = boto3.client('secretsmanager', region_name='us-east-1')
    RMD_KEY = os.environ.get('PSU_RESEARCH_API_KEY') or sm.get_secret_value(SecretId='overton/api-keys/rmd')['SecretString']
except Exception:
    RMD_KEY = os.environ['PSU_RESEARCH_API_KEY']

RMD_BASE = 'https://metadata.libraries.psu.edu/v1'

def rmd_get(path, params=None, auth=True):
    h = {'Accept': 'application/json'}
    if auth:
        h['X-API-Key'] = RMD_KEY
    r = requests.get(f'{RMD_BASE}{path}', params=params, headers=h, timeout=30)
    r.raise_for_status()
    time.sleep(0.3)
    return r.json()

def pretty(o, limit=2000):
    s = json.dumps(o, indent=2, default=str, ensure_ascii=False)
    print(s if len(s) < limit else s[:limit] + f'\n... ({len(s):,} chars total)')

print('ready')

## 1. `/organizations` — what units we have access to

Returns every PSU organization (department, center, college) the API key has visibility on.  Today: HHD-only — 17 units.

In [ ]:
orgs = rmd_get('/organizations')['data']
print(f'{len(orgs)} organizations\n')
for o in orgs:
    print(f'  {o["id"]:>4}  {o["attributes"]["name"]}')

## 2. `/organizations/{id}/publications` — the workhorse

Paginated list of every publication associated with an org.  This is what the pipeline scans across all 17 orgs to discover unique `psu_user_id`s and DOIs.  Pagination via `limit` + `offset`.

In [ ]:
sample_org = orgs[0]['id']  # College of HHD = id 10
pubs = rmd_get(f'/organizations/{sample_org}/publications', params={'limit': 3, 'offset': 0})['data']
print(f'Got {len(pubs)} publications for org {sample_org}\n')
print(f'Top-level publication attribute keys:')
print(' ', list(pubs[0]['attributes'].keys()))

In [ ]:
# A single publication record
pretty(pubs[0])

In [ ]:
# Contributor fields — this is where the pipeline gets psu_user_id, name, and sometimes ORCID
print('First contributor on first publication:')
pretty(pubs[0]['attributes']['contributors'][0])

In [ ]:
# How often do contributors carry an ORCID directly on the record?
batch = rmd_get(f'/organizations/{sample_org}/publications', params={'limit': 100})['data']
n_contrib = n_uid = n_orcid = 0
for p in batch:
    for c in p['attributes'].get('contributors', []):
        n_contrib += 1
        if c.get('psu_user_id'): n_uid += 1
        if c.get('orcid'):       n_orcid += 1
print(f'{n_contrib} contributor rows across {len(batch)} publications')
print(f'  with psu_user_id: {n_uid:>5}  ({n_uid/n_contrib:.0%})')
print(f'  with orcid:       {n_orcid:>5}  ({n_orcid/n_contrib:.0%})')

## 3. `/users/{uid}/profile` — researcher detail

Public endpoint (no API key required).  This is where `orcid_identifier`, organization name, Scopus metrics, and bio text live.

Note: `attributes.name` is the actual person name; `attributes.title` is their academic title (e.g. 'Professor').

In [ ]:
sample_uid = 'dhk102'  # Donna Korzick — reliably accessible
profile = rmd_get(f'/users/{sample_uid}/profile', auth=False)['data']
attrs = profile['attributes']
print(f'Profile attribute keys: {list(attrs.keys())}\n')
for k in ['name', 'title', 'organization_name', 'email', 'orcid_identifier',
         'total_scopus_citations', 'scopus_h_index', 'pure_profile_url']:
    v = attrs.get(k)
    print(f'  {k:<25} {str(v)[:90]!r}')

## 4. `/users/{uid}/grants` — funded research

Each grant has a title, agency, dollar amount, and date range.  Visibility is researcher-controlled — some users return empty arrays.

In [ ]:
grants = rmd_get(f'/users/{sample_uid}/grants')['data']
print(f'{len(grants)} grants for {sample_uid}')
if grants:
    print(f'Attribute keys: {list(grants[0]["attributes"].keys())}\n')
    pretty(grants[0])

## 5. `/users/{uid}/presentations`
Conference talks, posters, panels, etc.

In [ ]:
pres = rmd_get(f'/users/{sample_uid}/presentations')['data']
print(f'{len(pres)} presentations for {sample_uid}')
if pres:
    print(f'Attribute keys: {list(pres[0]["attributes"].keys())}')
    print()
    pretty(pres[0])

## 6. `/users/{uid}/organization_memberships`
Departments / centers a researcher belongs to, plus position title and dates.

In [ ]:
memberships = rmd_get(f'/users/{sample_uid}/organization_memberships')['data']
print(f'{len(memberships)} memberships')
for m in memberships:
    a = m['attributes']
    print(f'  {a.get("position_title"):<25}  {a.get("organization_name"):<35}  '
          f'{a.get("organization_type")}  start={a.get("position_started_on")}  end={a.get("position_ended_on")}')

## 7. `/users/{uid}/etds` — theses/dissertations supervised

In [ ]:
etds = rmd_get(f'/users/{sample_uid}/etds')['data']
print(f'{len(etds)} ETDs supervised')
if etds:
    pretty(etds[0])

## 8. `/publications/{id}` — single-publication detail
Richer than what the org-list returns: full abstract, tags, journal, contributor list.

In [ ]:
pub_id = pubs[0]['id']
pub = rmd_get(f'/publications/{pub_id}')['data']
print(f'Detail attribute keys: {list(pub["attributes"].keys())}\n')
for k in ['title', 'doi', 'journal_title', 'publisher', 'published_on', 'citation_count', 'publication_type', 'status']:
    v = pub['attributes'].get(k)
    print(f'  {k:<20} {str(v)[:90]!r}')

## 9. `/users/{uid}/publications` — per-user publications (with caveat)

**Warning:** this endpoint returns 404 for many users.  Visibility is researcher-configurable — it's not that the data is missing, it's that this endpoint respects per-user opt-in.  The pipeline does **not** rely on this endpoint — instead it uses the org-publications scan in section 2, which sees all contributors regardless of per-user visibility settings.

In [ ]:
# Compare two users — one accessible, one not
for uid in ['dhk102', 'jaf51']:
    try:
        d = rmd_get(f'/users/{uid}/publications', params={'limit': 3})
        print(f'  /users/{uid}/publications: {len(d["data"])} rows')
    except requests.HTTPError as e:
        print(f'  /users/{uid}/publications: HTTP {e.response.status_code}')

## What the pipeline pulls today (Stage 1: RMD)

Per researcher (looped over the cached ORCID→WebAccess map):

| Endpoint | Used for |
|---|---|
| `/users/{uid}/profile` | name, organization, email, Scopus metrics, bio |
| `/users/{uid}/grants` | grant list (title, agency, amount, dates) |
| `/users/{uid}/presentations` | count only (we don't store details) |
| `/users/{uid}/etds` | count only |
| `/users/{uid}/organization_memberships` | dept / center memberships |

Plus the one-time map-build (run via `--rebuild-map`) which scans `/organizations/{id}/publications` across all 17 orgs to discover all `psu_user_id`s and their associated DOIs.